<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 13 — Machine Learning Reference

A lookup guide. Notebooks 01 and 02 are meant to be read in order; this one is
meant to be **searched**. Keep it open while you work on Homework 12.

Every section runs on the small table `shops` defined in the first cell: 500
made-up restaurants in the shape of the homework's table. Run the setup cell,
then jump to the section you need.

| Section | When you need it |
|---|---|
| [A — The words](#A) | feature, target, training set, baseline, overfitting, ... |
| [B — Features and target](#B) | `X` and `y`; what may be a feature |
| [C — Turning columns into numbers](#C) | missing values, True/False, categories |
| [D — Splitting](#D) | by hand, with scikit-learn, stratified, by time |
| [E — The baseline](#E) | majority class; `DummyClassifier` |
| [F — A model: make, fit, predict, score](#F) | the four steps every model shares |
| [G — Decision trees](#G) | settings, reading a tree, what it used |
| [H — Counting right and wrong](#H) | confusion matrix, accuracy, precision, recall |
| [I — Overfitting, and choosing settings](#I) | train against test; validation; cross-validation |
| [J — Leakage](#J) | a checklist |
| [K — What to report](#K) | and what a model does not tell you |
| [L — Symptoms and causes](#L) | error messages and scores that look wrong |

**In Homework 12 you write the split (D), the baseline (E) and the scores (H)
yourself.** The scikit-learn versions are here so that you can check yours
against them in the notebook.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_score, recall_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

pd.options.display.float_format = "{:,.3f}".format

rng = np.random.default_rng(13)
ROWS = 500
shops = pd.DataFrame({
    "name": ["Shop {:03d}".format(n) for n in range(ROWS)],
    "state": rng.choice(["PA", "FL", "TN", "NV"], size=ROWS, p=[0.4, 0.3, 0.2, 0.1]),
    "stars": rng.choice(np.arange(1.0, 5.5, 0.5), size=ROWS),
    "review_count": np.maximum(5, rng.lognormal(3.5, 1.0, ROWS).round()).astype(int),
    "price": rng.choice([1.0, 2.0, 3.0, np.nan], size=ROWS, p=[0.4, 0.35, 0.1, 0.15]),
    "has_hours": rng.random(ROWS) < 0.85,
})
pull = -0.8 + 1.4 * (~shops["has_hours"]) + 0.5 * (shops["price"].fillna(0) >= 2) - 0.7 * (shops["review_count"] > 60)
shops["is_open"] = (rng.random(ROWS) >= 1 / (1 + np.exp(-pull))).astype(int)

print(shops.shape, " closed: {:.0%}".format(1 - shops["is_open"].mean()))
shops.head()

---

<a id="A"></a>
## A — The words

| Word | Means |
|---|---|
| **model** | a rule for making predictions, found by a program from examples |
| **feature** | a column the model may look at; together, `X` |
| **target** (label) | the column to predict; `y` |
| **classification** | the target is a category (open / closed) |
| **regression** | the target is a number (next month's sales) |
| **training set** | the rows the model learns from |
| **test set** | rows put aside, used once, to measure the finished model |
| **validation set** | rows taken from the training set to choose settings on |
| **fit** (train) | find the rule from the training set |
| **predict** | apply the rule to rows |
| **baseline** | the score of the simplest possible guess |
| **accuracy** | the share of predictions that are right |
| **positive class** | the class you are looking for, coded 1 |
| **precision** | of the rows predicted positive, the share that are |
| **recall** | of the rows that are positive, the share that were found |
| **confusion matrix** | the table of predicted against actual classes |
| **overfitting** | fitting the training rows, noise included, and so doing worse on new rows |
| **hyperparameter** | a setting chosen before training, such as a tree's depth |
| **leakage** | information in the features that would not be available when predicting |
| **imbalanced** | one class is much rarer than the other |

---

<a id="B"></a>
## B — Features and target

| You want | Write this |
|---|---|
| the features | `X = table[["stars", "review_count"]]` (a DataFrame: two brackets) |
| the target | `y = table["closed"]` (a Series: one bracket) |
| a target that is the reverse of a column | `y = (table["is_open"] == 0).astype(int)` |
| a target from a condition | `y = (table["stars"] >= 4).astype(int)` |
| how many rows and features | `X.shape` |
| how the classes are divided | `y.value_counts()`, `y.mean()` (the share of 1s) |
| do `X` and `y` belong together? | `X.index.equals(y.index)` |

A column may be a feature only if all of these hold:

- it is a **number** (after Section C), with **no missing value**;
- it is **not the target**, and not computed from it;
- it would be **known at the moment the prediction is needed**;
- it does not merely **identify the row** (an id, a name): a model can use
  such a column to memorise.

In [ ]:
y = (shops["is_open"] == 0).astype(int).rename("closed")
print(y.value_counts().to_dict(), " share closed: {:.3f}".format(y.mean()))

---

<a id="C"></a>
## C — Turning columns into numbers

| Column holds | Do this | Code |
|---|---|---|
| numbers, complete | nothing | |
| True / False | 1 / 0 | `table["has_hours"].astype(int)` |
| numbers with blanks | fill, **and** flag | `table["price"].fillna(0)` and `table["price"].notna().astype(int)` |
| a few categories | one 0/1 column each (one-hot) | `(table["state"] == "PA").astype(int)`, or `pd.get_dummies(table["state"], prefix="state").astype(int)` |
| many categories | group the rare ones first | `table["state"].where(table["state"].isin(common), "other")` |
| a date | parts of it | `table["date"].dt.year`, `.dt.month`, `.dt.dayofweek` |
| free text, ids, names | leave out | |

In [ ]:
X = shops[["stars", "review_count"]].copy()
X["price"] = shops["price"].fillna(0)
X["price_known"] = shops["price"].notna().astype(int)
X["has_hours"] = shops["has_hours"].astype(int)
for state in ("PA", "FL", "TN"):                      # a fixed list: the same columns every time
    X["state_" + state] = (shops["state"] == state).astype(int)

print(int(X.isna().sum().sum()), "missing values;", list(X.columns))
X.head(4)

- **Fill and flag.** A blank filled with 0 looks like a real 0. The flag
  column keeps the information that the value was missing, which is often
  informative.
- **Do not number categories** 1, 2, 3: that invents an order and distances.
- **Fix the list of categories.** `pd.get_dummies` makes columns for the
  values that happen to be present. Applied separately to two parts of the
  data it can produce different columns, and the model then fails or, worse,
  runs on misaligned columns. Loop over a fixed list, or call `get_dummies`
  once on the whole table before splitting.
- Trees do not care about the scale of a feature. Many other models do, and
  need the features scaled first.

---

<a id="D"></a>
## D — Splitting

In [ ]:
def split(X, y, test_share=0.25, seed=13):
    """A seeded random split, by hand."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(X))                    # positions, shuffled
    test_rows = int(round(len(X) * test_share))
    test, train = order[:test_rows], order[test_rows:]
    return X.iloc[train], X.iloc[test], y.iloc[train], y.iloc[test]


X_train, X_test, y_train, y_test = split(X, y)
print(len(X_train), len(X_test), " same rows in X and y:", X_test.index.equals(y_test.index))

| You want | Write this |
|---|---|
| a random split | `train_test_split(X, y, test_size=0.25, random_state=13)` |
| the same share of each class in both parts | `train_test_split(..., stratify=y)` |
| the order of the results | `X_train, X_test, y_train, y_test` |
| a split by time | `past = table["year"] <= 2017`; `X[past]`, `X[~past]` |
| a validation set | split the training part again |
| rows by position | `X.iloc[positions]` (keeps each row's index) |
| is every row in exactly one part? | `set(X_train.index) & set(X_test.index) == set()` |

In [ ]:
a_train, a_test, b_train, b_test = train_test_split(X, y, test_size=0.25, random_state=13)
c_train, c_test, d_train, d_test = train_test_split(X, y, test_size=0.25, random_state=13, stratify=y)

print("share closed -- all: {:.3f}   random split test: {:.3f}   stratified test: {:.3f}".format(
    y.mean(), b_test.mean(), d_test.mean()))

| Kind of split | Use when |
|---|---|
| random | rows are independent and the model will meet rows like these |
| stratified | a class is rare, and a random split could leave too few of it in one part |
| by time | the model will predict the future from the past |
| by group | several rows belong to one customer, patient or shop: keep each group in one part |

- Split **before** anything is learned from the data, including fill values
  and group averages.
- Always pass a seed, and report it.
- The split by hand, `train_test_split` and a classmate's split all put
  different rows in the test part. Scores will differ in the second decimal.
  That is the run-to-run variation of Session 10, not an error.

---

<a id="E"></a>
## E — The baseline

In [ ]:
majority = int(y_train.value_counts().idxmax())           # learned from the TRAINING part
baseline = float((y_test == majority).mean())             # measured on the TEST part
print("majority class:", majority, "  baseline accuracy: {:.3f}".format(baseline))

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("scikit-learn's version:              {:.3f}".format(dummy.score(X_test, y_test)))

| You want | Write this |
|---|---|
| the majority class | `y_train.value_counts().idxmax()` |
| its accuracy | `(y_test == majority).mean()` |
| the same as a model | `DummyClassifier(strategy="most_frequent")` |
| a guess in proportion to the classes | `DummyClassifier(strategy="stratified", random_state=13)` |

- Compute the baseline **first**, and put it in every table next to the
  model's score.
- With a rare class the baseline accuracy is high (97.5 % in notebook 02) and
  its recall is 0. Then compare recall and precision, not accuracy.
- A model that does not beat the baseline is a finding. Report it.

---

<a id="F"></a>
## F — A model: make, fit, predict, score

Every scikit-learn model is used the same way.

| Step | Code | Does |
|---|---|---|
| make | `model = DecisionTreeClassifier(max_depth=3, random_state=13)` | fixes the kind of model and its settings |
| fit | `model.fit(X_train, y_train)` | learns from the training part |
| predict | `model.predict(X_test)` | one class per row, as an array |
| score | `model.score(X_test, y_test)` | accuracy on those rows |
| chances | `model.predict_proba(X_test)[:, 1]` | the estimated chance of class 1, per row |

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=13)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(predictions[:15], type(predictions).__name__)
print("accuracy: {:.3f}   baseline: {:.3f}".format(model.score(X_test, y_test), baseline))
print("chance of 'closed' for the first five:", np.round(model.predict_proba(X_test)[:5, 1], 2))

- `predict` returns a plain numpy array with **no index**. To compare it with
  `y_test` (which has one), compare by position: `predictions ==
  y_test.to_numpy()`, or `np.asarray(...)` on both.
- `fit` must see the training part **only**.
- The columns given to `predict` must be the same, in the same order, as
  those given to `fit`.
- A new call of `fit` forgets everything from the previous one.

---

<a id="G"></a>
## G — Decision trees

| Setting | Means | Smaller value gives |
|---|---|---|
| `max_depth` | how many questions in a row | a simpler tree |
| `min_samples_leaf` | fewest training rows a verdict may rest on | (larger value) a simpler tree |
| `max_leaf_nodes` | most verdicts the tree may have | a simpler tree |
| `random_state` | the seed for breaking ties | — always set it |
| `class_weight="balanced"` | count the rare class more heavily | more predictions of the rare class |

| You want | Write this |
|---|---|
| the tree as text | `export_text(model, feature_names=list(X.columns))` |
| how many verdicts | `model.get_n_leaves()` |
| how deep it actually grew | `model.get_depth()` |
| which features it used, and how much | `model.feature_importances_` |
| a drawing | `sklearn.tree.plot_tree(model, feature_names=list(X.columns), ax=ax)` |

In [ ]:
print(export_text(model, feature_names=list(X.columns)))
print(model.get_n_leaves(), "leaves, depth", model.get_depth())
print(pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(4))

Reading a tree: each line is a question, `feature <= number`. Follow "yes"
down the first branch and "no" down the second. `class: 1` is the verdict.

- A question on a 0/1 column appears as `<= 0.50`: "is it 0?".
- A tree may ask about the same feature several times.
- A feature with importance 0 was not used at all.
- Importance says what the tree *used*, not what *causes* the outcome.

---

<a id="H"></a>
## H — Counting right and wrong

In [ ]:
def confusion(y_true, y_pred):
    true, pred = np.asarray(y_true), np.asarray(y_pred)        # by position
    return {"tp": int(((pred == 1) & (true == 1)).sum()),
            "fp": int(((pred == 1) & (true == 0)).sum()),
            "fn": int(((pred == 0) & (true == 1)).sum()),
            "tn": int(((pred == 0) & (true == 0)).sum())}


def scores(y_true, y_pred):
    c = confusion(y_true, y_pred)
    total = c["tp"] + c["fp"] + c["fn"] + c["tn"]
    return {"accuracy": (c["tp"] + c["tn"]) / total if total else np.nan,
            "precision": c["tp"] / (c["tp"] + c["fp"]) if c["tp"] + c["fp"] else np.nan,
            "recall": c["tp"] / (c["tp"] + c["fn"]) if c["tp"] + c["fn"] else np.nan}


print(confusion(y_test, predictions))
print({name: round(value, 3) for name, value in scores(y_test, predictions).items()})

| | Is 1 | Is 0 |
|---|---|---|
| **Predicted 1** | true positive, `tp` | false positive, `fp` (false alarm) |
| **Predicted 0** | false negative, `fn` (miss) | true negative, `tn` |

| Score | Formula | Question | Is NaN when |
|---|---|---|---|
| accuracy | (tp + tn) / all | how often right? | there are no rows |
| precision | tp / (tp + fp) | of those called 1, how many are? | nothing was called 1 |
| recall | tp / (tp + fn) | of those that are 1, how many were found? | nothing is 1 |

The same in scikit-learn. **Its confusion matrix is laid out differently**:
rows are the actual class, columns the predicted class, class 0 first.

In [ ]:
print(confusion_matrix(y_test, predictions))
print("   [[tn, fp],")
print("    [fn, tp]]")
print()
print("accuracy  {:.3f}".format(accuracy_score(y_test, predictions)))
print("precision {:.3f}".format(precision_score(y_test, predictions, zero_division=np.nan)))
print("recall    {:.3f}".format(recall_score(y_test, predictions, zero_division=np.nan)))

In [ ]:
print(classification_report(y_test, predictions, target_names=["open", "closed"]))

| You care most about | Watch | Example |
|---|---|---|
| not missing a case | recall | a smoke detector; a screening test |
| not raising false alarms | precision | a filter that deletes messages |
| both, with a rare class | precision and recall together, never accuracy alone | fraud; late answers |
| both, with balanced classes | accuracy, with the baseline beside it | |

Raising recall usually lowers precision, and the reverse. Predict 1 for
everything and recall is 100 %. The trade is made with the model's settings
or with a threshold on `predict_proba`: `(chances >= 0.3).astype(int)`
predicts 1 more readily than the default 0.5.

---

<a id="I"></a>
## I — Overfitting, and choosing settings

In [ ]:
rows = []
for depth in (1, 2, 3, 5, 8, 12, 20):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=13).fit(X_train, y_train)
    rows.append({"depth": depth, "leaves": tree.get_n_leaves(),
                 "train_accuracy": tree.score(X_train, y_train),
                 "test_accuracy": tree.score(X_test, y_test)})
pd.DataFrame(rows)

| What you see | Means |
|---|---|
| train and test accuracy both low, near the baseline | the model is too simple, or the features say little (*underfitting*) |
| train high, test clearly lower | overfitting |
| the gap grows as the model is allowed to grow | the usual picture |
| test higher than train | chance, with a small test set; or a mistake in the split |
| both near 100 % | suspect leakage before celebrating |

**Never choose a setting by its test score.** Choose on a validation set:

In [ ]:
X_fit, X_valid, y_fit, y_valid = split(X_train, y_train, seed=13)

results = {}
for depth in (1, 2, 3, 5, 8, 12, 20):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=13).fit(X_fit, y_fit)
    results[depth] = tree.score(X_valid, y_valid)

best = max(results.values())
chosen = min(depth for depth, score in results.items() if score == best)     # ties: the smaller tree
final = DecisionTreeClassifier(max_depth=chosen, random_state=13).fit(X_train, y_train)

print({depth: round(score, 3) for depth, score in results.items()})
print("chosen depth:", chosen, "  test accuracy, measured once: {:.3f}   baseline: {:.3f}".format(
    final.score(X_test, y_test), baseline))

| Rows | Used for | How often |
|---|---|---|
| fit part (of the training set) | training candidate models | as often as you like |
| validation part (of the training set) | choosing between them | as often as you like |
| test set | measuring the one chosen model | **once** |

**Cross-validation** makes the validation step steadier: it cuts the training
set into k parts and lets each be the validation part in turn.

In [ ]:
for depth in (2, 5, 20):
    folds = cross_val_score(DecisionTreeClassifier(max_depth=depth, random_state=13),
                            X_train, y_train, cv=5)
    print("depth {:2}: five validation accuracies {}   mean {:.3f}".format(
        depth, np.round(folds, 3), folds.mean()))

Look at how much the five numbers of one depth differ. A single split is one
sample; a difference of one or two points between two settings is often
smaller than that spread, and then the honest conclusion is "no real
difference; take the simpler model".

---

<a id="J"></a>
## J — Leakage

A feature leaks when it carries information that would not be available at
the moment a real prediction is needed.

| Kind | Example | Check |
|---|---|---|
| the target itself, or its mirror | `is_open` among the features when predicting `closed` | list the columns of `X` |
| a column recorded after the outcome | "response: untimely" when predicting lateness | when is each column filled in? |
| a statistic that includes the row | a company's late share computed over all rows | compute it from training rows only |
| a statistic that includes the test set | a fill value, an average, a list of categories taken from all rows | split first, then compute |
| the future | next month's value; a total over the whole year | would I have it on the day? |
| an identifier | an id that grows with time, or a name | leave identifiers out |
| tuning on the test set | picking the depth with the best test score | use a validation set |
| duplicates | the same row in both parts | drop duplicates before splitting |

Signs of a leak:

- a score far better than you would expect a person to manage;
- one feature with nearly all the importance, and a name that sounds like the
  target;
- a model that is excellent on the test set and poor on genuinely new data.

---

<a id="K"></a>
## K — What to report

| Item | Example |
|---|---|
| the task and the target | predict which restaurants are closed (1 = closed) |
| the data | 52,268 restaurants; 33 % closed |
| the features, and what was left out | 15 features; the response column was excluded as a leak |
| the split, with its seed | 75 % / 25 %, random, seed 8410 |
| the baseline | always "open": 67.3 % |
| how settings were chosen | depth chosen on a validation split of the training set |
| the result **on the test set** | accuracy 72.3 %; precision 0.63; recall 0.38 |
| how steady it is | within a point across five seeds |
| what it cannot do | says nothing about why restaurants close, or about new ones |

What a model does not tell you:

| It finds | It does not find |
|---|---|
| what goes together in the data it saw | what causes what |
| a rule that worked on the past | whether the future will resemble the past |
| the patterns of the people and places in the data | anything about those who are missing from it |
| a score for the average case | whether its mistakes fall evenly, or on one group |

---

<a id="L"></a>
## L — Symptoms and causes

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValueError: could not convert string to float: 'PA'` | a text column among the features | encode it (Section C) or leave it out |
| `ValueError: Input X contains NaN` | a missing value among the features | fill and flag (Section C) |
| `ValueError: The feature names should match those that were passed during fit` | other columns, or another order, at `predict` than at `fit` | build both parts with the same function and the same fixed list of categories |
| `ValueError: Found input variables with inconsistent numbers of samples` | `X` and `y` of different lengths | split them together, with the same positions |
| `X has 14 features, but DecisionTreeClassifier is expecting 15` | `get_dummies` applied separately to two parts | a fixed list of categories |
| training accuracy 100 % | an unlimited tree, on any data | not an error: look at the test accuracy |
| test accuracy 100 % or close | leakage | Section J |
| accuracy equal to the baseline; recall 0 | the model predicts the majority for every row | a rare class or weak features; look at the confusion matrix |
| accuracy far below the baseline | features and target scrambled against each other | check `X.index.equals(y.index)` after the split |
| a different result on every run | no `random_state` on the split or on the tree | set both |
| the same result whatever the seed | the seed is not passed on | follow it from the top |
| precision is `nan`, or an "ill-defined" warning | the model predicted no row as 1 | that is the answer; report recall 0 |
| scikit-learn's confusion matrix "has tp and tn swapped" | its layout is `[[tn, fp], [fn, tp]]` | read rows as actual, columns as predicted |
| everything all False when comparing `predictions == y_test` | comparing an array without index to a Series by label | `y_test.to_numpy()` |
| a setting chosen from the test table | the test set used twice | choose on a validation set |
| good on the test set, poor in use | a random split where the real use is "predict the future" | split by time |